# KM vazio histórico e entrada do replay (Gold)

Mede, para cada par de corridas consecutivas do mesmo motorista, o deslocamento sem passageiro entre o fim de uma corrida e o embarque da seguinte. É a **referência histórica** do KM morto contra a qual o Python compara a alocação preventiva D+1.

## Fluxo dos dados

* Origens: `tb_corridas` e `tb_deslocamentos_motoristas` da Silver
* Destinos em `cat_mobilidade_urbana_gold.db_frota`: `tb_km_vazio_segmentos`, `tb_km_vazio_diario`, `tb_corridas_replay`

## Parcelas medidas, sem definir o KPI corporativo (DT003, RN010, RN011)

Cada parcela é mutuamente exclusiva e fica em coluna própria; a empresa ainda não homologou quais compõem o KPI:

| Coluna | Trecho | Método |
|---|---|---|
| `km_reta_fim_ate_origem` | destino da corrida anterior → origem da seguinte | Haversine: **definição do PDF (p. 10)** |
| `km_reta_fim_ate_aceite` | destino anterior → posição no aceite | Haversine |
| `km_reta_aceite_ate_origem` | aceite → embarque | Haversine |
| `km_gps_ocioso` | `ONLINE`/`SELECIONADO` entre o fim anterior e o embarque | GPS amostrado |
| `km_gps_pickup_nao_concluido` | `A_CAMINHO` para solicitações que não são a corrida seguinte (ex.: canceladas) | GPS amostrado |
| `km_gps_pickup` | `A_CAMINHO` da corrida seguinte | GPS amostrado |

Segmentos com lacuna de GPS ou passagem por `INDISPONIVEL` saem de `fl_segmento_valido` (troca de turno, pausa ou offline não viram KM morto nem zero). Retorno à base não é observado e permanece ausente.

In [0]:
# No Databricks `spark` já existe; no VS Code o Databricks Connect cria uma sessão serverless.
try:
    spark  # noqa: F821
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless(True).getOrCreate()

from pyspark.sql import Window as W
from pyspark.sql import functions as F

# Todos os carimbos da fonte têm offset -03:00; datas e horas locais usam o fuso informado pelo proponente.
spark.conf.set("spark.sql.session.timeZone", "America/Sao_Paulo")

BRONZE = "cat_mobilidade_urbana_bronze.db_frota"
SILVER = "cat_mobilidade_urbana_silver.db_frota"
GOLD = "cat_mobilidade_urbana_gold.db_frota"


def segundos_entre(inicio, fim):
    """Diferença em segundos (com fração) entre dois timestamps."""
    return F.col(fim).cast("double") - F.col(inicio).cast("double")


def haversine_km(lat1, lon1, lat2, lon2):
    """Mesmo Haversine de src/tcc_frota/distance.py (R = 6371,0088 km).

    Linha reta entre dois pontos: estimativa geodésica, nunca rodagem observada (RN012).
    """
    la1, lo1, la2, lo2 = (F.radians(F.col(c)) for c in (lat1, lon1, lat2, lon2))
    h = F.pow(F.sin((la2 - la1) / 2), 2) + F.cos(la1) * F.cos(la2) * F.pow(F.sin((lo2 - lo1) / 2), 2)
    return F.lit(6371.0088) * 2 * F.asin(F.least(F.lit(1.0), F.sqrt(h)))


def pseudonimo(coluna):
    """Pseudônimo determinístico (SHA-256 truncado) para tabelas Gold; não é anonimização."""
    return F.substring(F.sha2(F.concat(F.lit("tcc-frota|"), F.col(coluna).cast("string")), 256), 1, 16)


def gravar(df, tabela, comentario):
    """Sobrescreve a tabela Delta de forma idempotente e registra o comentário."""
    df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
    spark.sql(f"COMMENT ON TABLE {tabela} IS '{comentario}'")
    print(f"{tabela}: {spark.table(tabela).count():,} linhas")

## 1. Pares de corridas consecutivas

In [0]:
corridas = spark.table(f"{SILVER}.tb_corridas").where("fl_tempos_ordenados")
o = W.partitionBy("id_motorista").orderBy("ts_aceite", "id_solicitacao")

pares = (
    corridas.select(
        "id_motorista",
        F.col("id_solicitacao").alias("id_corrida"),
        F.lag("id_solicitacao").over(o).alias("id_corrida_anterior"),
        F.lag("ts_fim").over(o).alias("ts_fim_anterior"),
        "ts_aceite", "ts_inicio", "ts_fim",
        F.lag("lat_destino").over(o).alias("lat_destino_anterior"),
        F.lag("lon_destino").over(o).alias("lon_destino_anterior"),
        F.lag("h3_r8_destino").over(o).alias("h3_r8_destino_anterior"),
        F.lag("fl_destino_confiavel").over(o).alias("fl_destino_anterior_confiavel"),
        "lat_aceite", "lon_aceite", "h3_r8_aceite", "fl_aceite_confiavel",
        "lat_origem", "lon_origem", "h3_r8_origem", "h3_r7_origem", "fl_origem_confiavel",
        "km_gps_pickup",
    )
    .where(F.col("id_corrida_anterior").isNotNull())
)

## 2. Trechos GPS após o fim da corrida anterior

Os fins de corrida entram como marcadores no fluxo de trechos de cada motorista. Cada trecho herda o último fim de corrida ocorrido antes dele, sem junção por intervalo.

In [0]:
fins = corridas.select("id_motorista", F.col("ts_fim").alias("ts_ordem"), F.col("id_solicitacao").alias("marcador"))
trechos = spark.table(f"{SILVER}.tb_deslocamentos_motoristas").select(
    "id_motorista",
    F.col("ts_snapshot_inicio").alias("ts_ordem"),
    "km_haversine", "categoria_km", "fl_passo_valido", "motivo_invalido", "id_solicitacao_inicio",
)
ordem_fluxo = (
    W.partitionBy("id_motorista")
    .orderBy("ts_ordem", F.col("marcador").isNull())
    .rowsBetween(W.unboundedPreceding, W.currentRow)
)
fluxo = (
    fins.unionByName(trechos, allowMissingColumns=True)
    .withColumn("id_corrida_anterior", F.last("marcador", ignorenulls=True).over(ordem_fluxo))
    .where(F.col("marcador").isNull() & F.col("id_corrida_anterior").isNotNull())
    .drop("marcador")
)

antes_embarque = F.col("ts_ordem") < F.col("ts_inicio")
valido = F.col("fl_passo_valido")
cat = F.col("categoria_km")

medidas = (
    fluxo.join(pares.select("id_motorista", "id_corrida_anterior", "id_corrida", "ts_inicio"),
               ["id_motorista", "id_corrida_anterior"])
    .groupBy("id_motorista", "id_corrida_anterior")
    .agg(
        F.sum(F.when(valido & antes_embarque & cat.isin("ONLINE", "SELECIONADO"), F.col("km_haversine")).otherwise(0.0)).alias("km_gps_ocioso"),
        F.sum(
            F.when(valido & antes_embarque & (cat == "A_CAMINHO") & ~F.coalesce(F.col("id_solicitacao_inicio") == F.col("id_corrida"), F.lit(False)),
                   F.col("km_haversine")).otherwise(0.0)
        ).alias("km_gps_pickup_nao_concluido"),
        F.sum((antes_embarque & (F.col("motivo_invalido") == "LACUNA_TEMPORAL")).cast("int")).alias("qtd_lacunas_gps"),
        F.sum((antes_embarque & (cat == "INDISPONIVEL")).cast("int")).alias("qtd_trechos_indisponivel"),
        F.sum((antes_embarque & ~valido).cast("int")).alias("qtd_trechos_invalidos"),
        F.sum(antes_embarque.cast("int")).alias("qtd_trechos"),
    )
)

## 3. Segmentos de KM vazio

In [0]:
segmentos = (
    pares.join(medidas, ["id_motorista", "id_corrida_anterior"], "left")
    .fillna(0, subset=["km_gps_ocioso", "km_gps_pickup_nao_concluido", "qtd_lacunas_gps",
                       "qtd_trechos_indisponivel", "qtd_trechos_invalidos", "qtd_trechos"])
    .withColumn("min_fim_ate_aceite", F.round(segundos_entre("ts_fim_anterior", "ts_aceite") / 60, 2))
    .withColumn("min_fim_ate_embarque", F.round(segundos_entre("ts_fim_anterior", "ts_inicio") / 60, 2))
    .withColumn("fl_aceite_antes_fim_anterior", F.col("ts_aceite") < F.col("ts_fim_anterior"))
    .withColumn("km_reta_fim_ate_origem", haversine_km("lat_destino_anterior", "lon_destino_anterior", "lat_origem", "lon_origem"))
    .withColumn("km_reta_fim_ate_aceite", haversine_km("lat_destino_anterior", "lon_destino_anterior", "lat_aceite", "lon_aceite"))
    .withColumn("km_reta_aceite_ate_origem", haversine_km("lat_aceite", "lon_aceite", "lat_origem", "lon_origem"))
    .withColumn("km_gps_pickup", F.coalesce("km_gps_pickup", F.lit(0.0)))
    .withColumn("km_gps_vazio_total", F.col("km_gps_ocioso") + F.col("km_gps_pickup_nao_concluido") + F.col("km_gps_pickup"))
    .withColumn("fl_mesma_sessao", (F.col("qtd_lacunas_gps") == 0) & (F.col("qtd_trechos_indisponivel") == 0))
    .withColumn(
        "fl_posicoes_confiaveis",
        F.coalesce(F.col("fl_destino_anterior_confiavel"), F.lit(False))
        & F.col("fl_aceite_confiavel") & F.col("fl_origem_confiavel"),
    )
    .withColumn("fl_segmento_valido", F.col("fl_mesma_sessao") & F.col("fl_posicoes_confiaveis"))
    .withColumn("data_local", F.to_date("ts_inicio"))
    .withColumn("hora_local", F.hour("ts_inicio"))
    .withColumn("dia_semana_iso", ((F.dayofweek("ts_inicio") + 5) % 7) + 1)
    .withColumn("id_motorista_pseud", pseudonimo("id_motorista"))
    .withColumn("metodo_distancia", F.lit("haversine_reta_e_gps_amostrado_2min"))
    .drop("id_motorista", "lat_destino_anterior", "lon_destino_anterior", "lat_aceite", "lon_aceite",
          "lat_origem", "lon_origem")
)
gravar(
    segmentos,
    f"{GOLD}.tb_km_vazio_segmentos",
    "KM sem passageiro entre corridas consecutivas do mesmo motorista, em parcelas separadas (reta e GPS amostrado). "
    "Referencia historica; KPI corporativo pendente (DT003).",
)

## 4. Resumo diário

A data do segmento é a do embarque da corrida seguinte (`ts_inicio`); em `tb_demanda_*` a data é a do registro da solicitação.
Totais de segmentos válidos e, separadamente, a quilometragem GPS de toda a frota por estado. Taxas e reduções devem ser calculadas no Python sobre o mesmo recorte de cenários (RN028).

In [0]:
seg = spark.table(f"{GOLD}.tb_km_vazio_segmentos")
diario_segmentos = seg.groupBy("data_local").agg(
    F.count("*").alias("qtd_segmentos"),
    F.sum(F.col("fl_segmento_valido").cast("int")).alias("qtd_segmentos_validos"),
    *[
        F.round(F.sum(F.when(F.col("fl_segmento_valido"), F.col(c))), 3).alias(f"{c}_validos")
        for c in ("km_reta_fim_ate_origem", "km_reta_fim_ate_aceite", "km_reta_aceite_ate_origem",
                  "km_gps_ocioso", "km_gps_pickup_nao_concluido", "km_gps_pickup", "km_gps_vazio_total")
    ],
)

frota = (
    spark.table(f"{SILVER}.tb_deslocamentos_motoristas")
    .groupBy("data_local")
    .agg(
        *[
            F.round(F.sum(F.when(F.col("fl_passo_valido") & (F.col("categoria_km") == e), F.col("km_haversine"))), 3)
            .alias(f"km_gps_frota_{e.lower()}")
            for e in ("ONLINE", "SELECIONADO", "A_CAMINHO", "COM_PASSAGEIRO", "INDISPONIVEL")
        ],
        F.sum((~F.col("fl_passo_valido")).cast("int")).alias("qtd_trechos_invalidos_frota"),
        F.countDistinct("id_motorista").alias("qtd_motoristas_com_gps"),
    )
)
atendidas = (
    spark.table(f"{SILVER}.tb_corridas")
    .groupBy("data_local")
    .agg(
        F.count("*").alias("qtd_corridas_atendidas"),
        F.sum(F.col("fl_corrida_espacial_valida").cast("int")).alias("qtd_corridas_espaciais_validas"),
    )
)
# A janela observada é a do GPS da frota; agendamentos executados após o fim da extração ficam de fora.
diario = (
    frota.join(atendidas, "data_local", "left")
    .join(diario_segmentos, "data_local", "left")
    .withColumn("dia_semana_iso", ((F.dayofweek("data_local") + 5) % 7) + 1)
    .orderBy("data_local")
)
gravar(
    diario,
    f"{GOLD}.tb_km_vazio_diario",
    "Resumo diario: parcelas de KM vazio dos segmentos validos e KM GPS da frota por estado. Estimativas, nao odometro.",
)

## 5. Entrada do replay em Python

Uma linha por corrida atendida com origem e destino confiáveis. Coordenadas arredondadas a 3 casas (~110 m) para reduzir risco de reidentificação de endereços; os cálculos de KM da Silver usaram a precisão original.

In [0]:
replay = (
    spark.table(f"{SILVER}.tb_corridas")
    .where("fl_corrida_espacial_valida")
    .select(
        F.col("id_solicitacao").alias("id_corrida"),
        pseudonimo("id_motorista").alias("id_motorista_pseud"),
        pseudonimo("id_veiculo").alias("id_veiculo_pseud"),
        "tipo_solicitacao", "ts_registro", "ts_aceite", "ts_chegada", "ts_inicio", "ts_fim",
        "data_local", "hora_local", "dia_semana_iso", "fl_fim_de_semana",
        *[F.round(c, 3).alias(c) for c in ("lat_aceite", "lon_aceite", "lat_origem", "lon_origem", "lat_destino", "lon_destino")],
        "h3_r8_origem", "h3_r7_origem", "h3_r8_destino", "h3_r7_destino",
        "km_pickup_reta", "km_viagem_reta", "km_gps_pickup", "km_gps_viagem",
        (F.col("seg_viagem") / 60).alias("min_viagem"),
        "fl_aceite_confiavel",
    )
)
gravar(
    replay,
    f"{GOLD}.tb_corridas_replay",
    "Corridas atendidas com origem e destino confiaveis para o replay em Python. Coordenadas com 3 casas decimais; "
    "motorista e veiculo pseudonimizados.",
)